In [1]:
import pandas as pd
print("Pandas version:", pd.__version__)

Pandas version: 3.1.0


## Goal: Predict at purchase time whether an Olist order will be delivered later than the estimated date.

### Load everything (notebook notebooks/01_understanding.ipynb)


In [2]:
import pandas as pd 
from pathlib import Path

RAW = Path("../data/raw")

files = { "Customers": "olist_customers_dataset.csv",
          "Geolocation": "olist_geolocation_dataset.csv",
          "Items": "olist_order_items_dataset.csv",
          "Payments": "olist_order_payments_dataset.csv",
          "Reviews": "olist_order_reviews_dataset.csv",
          "Orders": "olist_orders_dataset.csv",
          "Products": "olist_products_dataset.csv",
          "Sellers": "olist_sellers_dataset.csv",
          "Category": "product_category_name_translation.csv"
        }
dfs = { name : pd.read_csv(RAW/f) for name , f in files.items() }
for name, df in dfs.items():
    print(f"{name:22} rows={df.shape[0]:>7} cols={df.shape[1]}")
    

Customers              rows=  99441 cols=5
Geolocation            rows=1000163 cols=5
Items                  rows= 112650 cols=7
Payments               rows= 103886 cols=5
Reviews                rows=  99224 cols=7
Orders                 rows=  99441 cols=8
Products               rows=  32951 cols=9
Sellers                rows=   3095 cols=4
Category               rows=     71 cols=2


In [3]:
for name,df in dfs.items():
    print(f"\n===== {name} =====")
    display(df.head(3))
    df.info()


===== Customers =====


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

===== Geolocation =====


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB

===== Items =====


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


... (9 outputs left out)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB

===== Category =====


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


### Code to verify

In [4]:
orders = dfs["Orders"]; items = dfs["Items"]; pay = dfs["Payments"]
rev = dfs["Reviews"]; cust = dfs["Customers"]; geo = dfs["Geolocation"]

# 1. Is the key unique? (grain check)
print("orders order_id unique:", orders["order_id"].is_unique)
print("items rows / unique orders:", len(items), items["order_id"].nunique())
print("payments rows / unique orders:", len(pay), pay["order_id"].nunique())
print("reviews rows / unique orders:", len(rev), rev["order_id"].nunique())
print("customers_id vs customer_unique_id:", cust["customer_id"].nunique(), cust["customer_unique_id"].nunique())
print("geo duplicate rows:", geo.duplicated().sum(), "| unique zip:", geo["geolocation_zip_code_prefix"].nunique())

# 2. Missing values per table
for name, df in dfs.items():
    nulls = df.isnull().sum()
    print(f"\n{name}:\n{nulls[nulls >0]}")

# 3. Order status
print(orders["order_status"].value_counts())

orders order_id unique: True
items rows / unique orders: 112650 98666
payments rows / unique orders: 103886 99440
reviews rows / unique orders: 99224 98673


customers_id vs customer_unique_id: 99441 96096
geo duplicate rows: 261831 | unique zip: 19015

Customers:
Series([], dtype: int64)

Geolocation:
Series([], dtype: int64)

Items:
Series([], dtype: int64)

Payments:
Series([], dtype: int64)

Reviews:
review_comment_title      87656
review_comment_message    58247
dtype: int64

Orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Products:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Sellers:
Series([], dtype: int64)

Category:
Series([], dtype: int64)
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved  

In [5]:
orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

## 3. Data cleaning

### 3A. Convert dates

In [12]:
orders = dfs["Orders"].copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### 3B. Apply the scope decision

In [17]:
orders = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].notnull())
].copy()
print(len(orders))  # about 96,470

96470


Why: The target needs a real delivery date.

### 3C. Create the target and time features

In [20]:
orders["is_late"] = (
    orders["order_delivered_customer_date"] > orders["order_estimated_delivery_date"]
).astype(int)

orders["delivery_date"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days
orders["delivery_estimate"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

print(orders["is_late"].value_counts(normalize=True))
print(orders[["delivery_date", "delivery_estimate"]].describe())

is_late
0    0.918876
1    0.081124
Name: proportion, dtype: float64
       delivery_date  delivery_estimate
count   96470.000000       96470.000000
mean       12.093604          23.372748
std         9.551380           8.758421
min         0.000000           2.000000
25%         6.000000          18.000000
50%        10.000000          23.000000
75%        15.000000          28.000000
max       209.000000         155.000000


### 3D. Remove duplicates by grain

In [23]:
# Reviews: keep the latest review per order
rev = dfs["Reviews"].copy()
rev["review_creation_date"] = pd.to_datetime(rev["review_creation_date"])
rev = rev.sort_values("review_creation_date").drop_duplicates("order_id", keep="last")
print(len(rev),rev["order_id"].nunique()) # must be equal

# Geolocation: one row per zip prefix (median is robust to bad points)
geo = (dfs["Geolocation"]
       .groupby("geolocation_zip_code_prefix")
       .agg(lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"))
       .reset_index()
)
print(len(geo)) # about 19,000


98673 98673
19015


### 3E. Products: translate categories and handle nulls

In [28]:
prod = dfs["Products"].merge(dfs["Category"], on="product_category_name", how="left")
prod["Category"] = (prod["product_category_name_english"]
                    .fillna(prod["product_category_name"])
                    .fillna("unknown"))
print(prod[["product_weight_g", "product_length_cm" ]].isnull().sum())

product_weight_g     2
product_length_cm    2
dtype: int64


3F. Sanity checks and outliers

In [32]:
# `delivery_days` is the same metric as the earlier `delivery_date` feature
if "delivery_days" not in orders.columns:
    orders["delivery_days"] = orders["delivery_date"].copy()

print("Negative delivery days:", (orders["delivery_days"] < 0).sum())
print("Over 100 days:", (orders["delivery_days"] > 100).sum())
print("Delivered before carrier pickup:",
      (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum())

Negative delivery days: 0
Over 100 days: 63
Delivered before carrier pickup: 23


3G. Save and commit

In [33]:
orders.to_csv("../data/processed/orders_clean.csv", index=False)
rev.to_csv("../data/processed/reviews_clean.csv", index=False)
geo.to_csv("../data/processed/geo_clean.csv", index=False)
prod.to_csv("../data/processed/products_clean.csv", index=False)

## Step 4: Joins and the one-row-per-order table

ML needs one flat table where each row is one order. Items, payments, and products have many rows per order, so we aggregate first, then join. 
`If skipped`: Joining raw tables multiplies rows, so one order appears many times and revenue and labels are wrong.


4A. Aggregate to the order grain


In [36]:
items, pay, cust, sellers = dfs["Items"], dfs["Payments"], dfs["Customers"], dfs["Sellers"]

# items + product info -> one row per order
it = items.merge(
    prod[[
        "product_id",
        "Category",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ]].rename(columns={"Category": "category"}),
    on="product_id",
    how="left",
)
it["volume_cm3"] = it["product_length_cm"] * it["product_height_cm"] * it["product_width_cm"]

it_agg = it.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_weight_g=("product_weight_g", "sum"),
    total_volume_cm3=("volume_cm3", "sum"),
    seller_id=("seller_id", "first"),
    category=("category", "first"),
).reset_index()

# payments -> one row per order
pay_agg = pay.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
).reset_index()
pay_type = (pay.sort_values("payment_value", ascending=False)
              .drop_duplicates("order_id")[["order_id", "payment_type"]])

4B. Join with a safety check (left joins only)

In [44]:
n0 = len(orders)
base_cols = ["order_id", "customer_id", "order_purchase_timestamp",
             "estimated_days", "is_late", "delivery_days"]

df = orders[base_cols].copy()

# List of merges to perform
merges = [
    (cust[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]], "customer_id"),
    (it_agg, "order_id"),
    (pay_agg, "order_id"),
    (pay_type, "order_id"),
    (sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], "seller_id"),
]

for right, key in merges:
    # Check if we are joining on seller_id after it has been brought in by items
    if key == "seller_id":
        # validate="many_to_one" ensures 'seller_id' is unique in the sellers dataframe,
        # allowing multiple orders to map to the same seller without crashing.
        df = df.merge(right, on=key, how="left", validate="many_to_one")
    else:
        # For order_id and customer_id, it should ideally be a 1:1 match per order row
        df = df.merge(right, on=key, how="left", validate="one_to_one")

# Final sanity check on the ultimate row count
assert len(df) == n0, f"Final row count ({len(df)}) does not match original orders ({n0})!"

print(df.shape)


(96470, 22)


Why left join: An inner join silently drops orders that lack a match, so you lose data without noticing.
Why the assert: It stops the code the moment a join duplicates rows.


4C. Add geography and distance

In [47]:
g = geo.rename(columns={"geolocation_zip_code_prefix": "zip"})
df = (df.merge(g.rename(columns={"zip": "customer_zip_code_prefix", "lat": "cust_lat", "lng": "cust_lng"}),
               on="customer_zip_code_prefix", how="left")
        .merge(g.rename(columns={"zip": "seller_zip_code_prefix", "lat": "sell_lat", "lng": "sell_lng"}),
               on="seller_zip_code_prefix", how="left"))
assert len(df) == n0

import numpy as np
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df.sell_lat, df.sell_lng, df.cust_lat, df.cust_lng)
df["same_state"] = (df["customer_state"] == df["seller_state"]).astype(int)

Why distance: Farther delivery means a higher chance of delay. This is likely one of your strongest features.

4D. Time features (known at purchase)

In [48]:
t = df["order_purchase_timestamp"]
df["purchase_month"]   = t.dt.month
df["purchase_weekday"] = t.dt.weekday
df["purchase_hour"]    = t.dt.hour

C:\Users\user\AppData\Local\Temp\ipykernel_11872\1224113992.py:3: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["purchase_weekday"] = t.dt.weekday


4E. Review score for EDA only

In [ ]:
df_eda = df.merge(rev[["order_id", "review_score"]], on="order_id", how="left")
print(df_eda)

                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
96465  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
96466  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
96467  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
96468  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
96469  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_purchase_timestamp  estimated_days  is_late  delivery_days  \
0

Leakage warning: A review is written after delivery. It answers question 3 (does lateness hurt reviews?) but is never a model feature. The same applies to delivery_days.


4F. Check and save

In [51]:
print(df.isnull().sum()[lambda s: s > 0])
print(df["is_late"].mean())

df.to_csv("../data/processed/model_table.csv", index=False)
df_eda.to_csv("../data/processed/eda_table.csv", index=False)

payment_value         1
max_installments      1
payment_type          1
cust_lat_x          264
cust_lng_x          264
sell_lat_x          215
sell_lng_x          215
distance_km         478
cust_lat_y          264
cust_lng_y          264
sell_lat_y          215
sell_lng_y          215
cust_lat            264
cust_lng            264
sell_lat            215
sell_lng            215
dtype: int64
0.08112366538820359
